# Rectified Flow Inversion — Stable Audio 3

RF-Inversion ([Rout et al., 2024](https://arxiv.org/abs/2410.10792)) on
[`stabilityai/stable-audio-3-medium-base`](https://huggingface.co/stabilityai/stable-audio-3-medium-base) —
the un-post-trained rectified-flow checkpoint, which is the one that inverts cleanly.

The flow ODE runs `t = 1` (noise) → `t = 0` (data). Inversion is the same ODE integrated the
other way. The two optional controllers (`gamma`, `eta`) are linear blends in velocity space;
with both at `0` you get plain deterministic inversion, which should reconstruct the input.

Implementation lives in `rf_inversion.py`.

## Setup

In [1]:
from pathlib import Path

import torch
import torchaudio
import IPython.display as ipd

from stable_audio_3.model import StableAudioModel
from stable_audio_3.inference.audio_utils import prepare_audio

from rf_inversion import invert, sample, invert_and_edit, make_cond

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention
No module named 'flash_attn'
flash_attn varlen/bert_padding not available, disabling varlen attention
device: mps


In [2]:
# "medium-base"      1.4B, ~9.2 GB fp32, best quality
# "small-music-base" 433M, ~2.1 GB, music only, several times faster
# "small-sfx-base"   433M, ~2.1 GB, sound effects only
# Use a *-base checkpoint: the post-trained ones are distilled for few-step
# ping-pong sampling and do not invert cleanly.
MODEL_ID = "small-music-base"

model = StableAudioModel.from_pretrained(MODEL_ID, device=device)

SR = model.model.sample_rate                  # 44100
DS = model.same.downsampling_ratio            # 4096 -> ~10.8 latent frames/sec

print(f"{MODEL_ID}: {sum(p.numel() for p in model.model.model.parameters()) / 1e6:.0f}M params")
print("objective:", model.model.diffusion_objective)
print("sample rate:", SR, "| downsampling:", DS, "| latent channels:", model.model.io_channels)

/Users/tom/Documents/ML/Models/sa3-inversion/.venv/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


small-music-base: 459M params
objective: rectified_flow
sample rate: 44100 | downsampling: 4096 | latent channels: 256


## Audio helpers

In [3]:
AUDIO_DIR = Path("audio")          # sources live here; generated clips are written here too
AUDIO_DIR.mkdir(exist_ok=True)


def load_latent(path, seconds=10.0, offset=0.0):
    """Load audio -> normalised stereo tensor + latent. Length is snapped to a multiple of DS."""
    wav, sr = torchaudio.load(AUDIO_DIR / path)
    wav = wav[:, int(offset * sr):]
    n = (int(seconds * SR) // DS) * DS
    audio = prepare_audio(wav, in_sr=sr, target_sr=SR, target_length=n, target_channels=2, device=device)
    audio = audio / audio.abs().max().clamp(min=1e-6)
    latent = model.same.encode(audio.to(next(model.same.parameters()).dtype))
    return audio, latent, n / SR


def to_audio(latent):
    return model.same.decode(latent).float().clamp(-1, 1).cpu()


def play(audio, label=None, name=None):
    """`name` is a bare filename; everything is written under AUDIO_DIR."""
    audio = audio[0] if audio.dim() == 3 else audio
    audio = (audio / audio.abs().max().clamp(min=1e-6)).float().cpu()
    if name:
        torchaudio.save(str(AUDIO_DIR / name), audio, SR)
    if label:
        print(label)
    ipd.display(ipd.Audio(audio.numpy(), rate=SR))

In [4]:
AUDIO_PATH = "rain_short.wav"
N_SEC = 10.0

audio, latent, seconds_total = load_latent(AUDIO_PATH, seconds=N_SEC)
print("audio:", tuple(audio.shape), "-> latent:", tuple(latent.shape), f"({seconds_total:.2f}s)")

play(audio, "Reference:", "reference.wav")

audio: (1, 2, 438272) -> latent: (1, 256, 108) (9.94s)
Reference:


## 1. Autoencoder round-trip

Before blaming the inversion for anything, check what the SAME autoencoder alone gives back.
This is the ceiling on reconstruction quality.

In [5]:
play(to_audio(latent), "Autoencoder round-trip:", "ae_roundtrip.wav")

Autoencoder round-trip:


## 2. Inversion sanity check

`gamma = 0`, `eta = 0`, empty prompt, no CFG — deterministic inversion followed by deterministic
re-sampling over the same schedule.

**`fixed_point_iters`.** `sample` steps down from `t_next` evaluating `v` there, so a plain Euler
step *up* — which evaluates at `t_curr` — is not its inverse. The mismatch compounds: plain Euler
reconstructs at 0.63 rel-err and doubling the steps only reaches 0.48. Solving the sampler's step
implicitly instead, `z <- y + dt·v(z, t_next)`, makes inversion an exact inverse. 2 iterations is
the sweet spot; more does not help.

**Schedule.** `"model"` is SA3's own generation spacing, heavily noise-dense — at 25 steps `dt` is
0.0028 next to `t=1` but 0.158 next to `t=0`. You might expect inversion to want the opposite, but
measured on medium-base it is the best of the options by a wide margin:

| schedule | model | logsnr | t² | t^1.5 | linear |
|---|---|---|---|---|---|
| recon rel-err | **0.152** | 0.293 | 0.345 | 0.516 | 0.569 |

Steps matter more than spacing. Both passes must use the same schedule.

| steps | 25 | 50 | 100 |
|---|---|---|---|
| medium-base | 0.154 | **0.018** | 0.007 |
| small-music-base | 0.153 | **0.026** | — |

In [6]:
STEPS = 50

SCHEDULE = "model"    # or "logsnr" / "linear"
FP_ITERS = 2          # 0 = plain Euler, which reconstructs poorly

cond = make_cond(model, "", seconds_total, latent.shape[-1])

inverted = invert(model, latent, cond, steps=STEPS, gamma=0.0, schedule=SCHEDULE, fixed_point_iters=FP_ITERS)
recon = sample(model, inverted, cond, steps=STEPS, cfg_scale=1.0, eta=0.0, schedule=SCHEDULE)

rel_err = ((recon - latent).norm() / latent.norm()).item()
print(f"inverted noise: mean {inverted.mean():+.3f}  std {inverted.std():.3f}  (N(0,1) if it were pure noise)")
# Content-dependent. On a 10s SFX-ambience clip at 50 steps: ~0.018 medium-base,
# ~0.026 small-music-base. Plain Euler (FP_ITERS=0) would be ~0.5.
print(f"latent rel-err: {rel_err:.4f}")

play(to_audio(recon), "Reconstruction:", "reconstructed.wav")
play(audio, "Reference (for comparison):")

inverted noise: mean +0.007  std 1.068  (N(0,1) if it were pure noise)
latent rel-err: 0.0187
Reconstruction:


Reference (for comparison):


## 3. Editing

Invert under an empty prompt, then re-sample under a target prompt.

- `cfg_scale` — how hard the new prompt pushes.
- `eta` — pull back toward the source latent during sampling. Higher keeps more of the original.
- `start` / `stop` — the fraction of the trajectory over which `eta` is active. Structure is decided
  early (high `t`), so leaving the controller off at the start lets the prompt reshape the piece.
- `gamma` — pulls the inversion toward Gaussian noise. Non-zero trades reconstruction fidelity for
  a starting point that behaves more like a normal sample.

In [7]:
edit_prompts = [
    "a drum breakbeat",
]

for i, prompt in enumerate(edit_prompts):
    print(f"--- {prompt!r} ---")
    edited, _ = invert_and_edit(
        model,
        latent,
        target_prompt=prompt,
        seconds_total=seconds_total,
        inversion_steps=STEPS,
        sampling_steps=STEPS,
        gamma=0.0,
        eta=0.4,
        start=0.0,
        stop=0.9,
        cfg_scale=6.0,
        schedule=SCHEDULE,
        fixed_point_iters=FP_ITERS,
    )
    play(to_audio(edited), None, f"edited_{i + 1}.wav")

--- 'a drum breakbeat' ---


## 4. Parameter sweep

`gamma` steers the inversion (forward pass); `eta` steers the sampling (reverse pass). So the
inversion is independent of `eta`, `cfg_scale`, and the target prompt — invert once, then re-sample
as many times as you like. `sample` copies its input, so the same `inverted` tensor is reusable.

Sweeping `gamma`, `steps`, `schedule`, or the *source* prompt would require re-inverting each time.

In [8]:
PROMPT = "a drum breakbeat"

src_cond = make_cond(model, "", seconds_total, latent.shape[-1])
inverted = invert(model, latent, src_cond, steps=STEPS, gamma=0.0,
                  schedule=SCHEDULE, fixed_point_iters=FP_ITERS)
edit_cond = make_cond(model, PROMPT, seconds_total, latent.shape[-1])

for eta, cfg in [(0.0, 6.0), (0.1, 6.0), (0.2, 6.0), (0.3, 6.0), (0.4, 6.0), (0.5, 6.0)]:
    out = sample(
        model, inverted, edit_cond,
        steps=STEPS, cfg_scale=cfg, eta=eta, source_latent=latent, stop=0.9,
        schedule=SCHEDULE, disable_tqdm=True,
    )
    play(to_audio(out), f"eta={eta}  cfg={cfg}", f"sweep_eta{eta}_cfg{cfg}.wav")

eta=0.0  cfg=6.0


eta=0.1  cfg=6.0


eta=0.2  cfg=6.0


eta=0.3  cfg=6.0


eta=0.4  cfg=6.0


eta=0.5  cfg=6.0
